# Data Schema Exploration — Graph-RAG for Energy Infrastructure Maintenance

**BITS Pilani dissertation** · *Graph-RAG Architectures for Grounded Decision Support in Massive Energy Infrastructure Maintenance (Shell Context)*

This notebook visualises the **generated data schema** — both the **structured** property graph (assets, components, telemetry, failure modes, SOPs, work orders and their relationships) and the **unstructured** text corpus (SOP bodies, work-order summaries).

## What you'll see
1. **Ontology / meta-schema** — the 6 node labels and 6 relationship types
2. **Knowledge graph** — the actual generated graph, nodes colored by label
3. **Asset topology** — the `CONNECTED_TO` process-flow per platform (the structure the Graph-RAG reasons over)
4. **Structured distributions** — component types, status, failure modes, priorities, telemetry
5. **Unstructured text overview** — document length by source type + top corpus terms
6. **Interactive exploration** — slice the data and build custom views

> **Prerequisites:** run `python -m scripts.generate_data` once (from the repo root) so the dataset exists, and `pip install matplotlib networkx pandas` for the plots. This notebook should be launched from the repo root (`jupyter lab` / `jupyter notebook`).

## 0. Setup — make the `graphrag` package importable

The notebook lives in `notebooks/`, so we add the project's `src/` to the path. We also generate the dataset automatically if it hasn't been created yet.

In [ ]:
import sys, os
from pathlib import Path

# repo root = parent of the notebooks/ folder
ROOT = Path.cwd()
if (ROOT / "notebooks").exists() is False and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent  # launched from inside notebooks/
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

print("repo root:", ROOT)
print("src on path:", SRC.exists())

In [ ]:
from graphrag import config as cfgmod
from graphrag.config import CONFIG
from graphrag import pipeline

# Generate the dataset if it is missing (idempotent, reproducible via the seed).
if not cfgmod.DATASET_FILE.exists():
    print("Dataset not found — generating it now...")
    pipeline.generate_and_save(CONFIG)

dataset = pipeline.load_dataset()
corpus = pipeline.load_corpus()
print("Loaded dataset with counts:")
dataset["metadata"]["counts"]

In [ ]:
# Build the in-memory knowledge graph from the dataset.
from graphrag.graph_store import get_graph_store
from graphrag.ingest import build_graph
from graphrag import visualization as viz

store = get_graph_store(CONFIG.graph)
build_graph(store, dataset)
print(f"Graph: {len(store.nodes())} nodes, {len(store.edges())} edges, backend={CONFIG.graph.backend}")

## 1. Ontology / meta-schema

The ISO 14224-adapted property-graph ontology: what kinds of things exist and how they relate. This is the *schema of schema types* — independent of any specific data.

In [ ]:
fig = viz.plot_ontology()
plt.show()

## 2. The knowledge graph (structured)

The actual generated graph. Nodes are colored by label; node size scales with degree. Telemetry leaves are excluded by default for legibility — pass `exclude_labels=[]` to include everything.

In [ ]:
fig = viz.plot_knowledge_graph(store)
plt.show()

## 3. Asset process-flow topology (structured)

The `CONNECTED_TO` edges for a single platform, rooted at its Separator. This is exactly the structure that lets Graph-RAG answer *"failure modes for pumps **downstream** of Separator V-202"* — something naive vector RAG cannot traverse.

In [ ]:
fig = viz.plot_asset_topology(store)   # auto-picks the richest platform; pass asset_id=... to choose
plt.show()

## 4. Structured attribute distributions

Node counts by label, component types and status, work orders per failure mode, priority mix, and the telemetry vibration distribution (with the 4.5 mm/s alarm threshold from failure mode FM-01).

In [ ]:
fig = viz.plot_schema_distributions(dataset)
plt.show()

## 5. Unstructured text corpus overview

The corpus is the text side of the dataset — SOP bodies, work-order summaries, component descriptions. Here we look at document length by source type and the most frequent domain terms (stopwords removed). This is the material the vector index embeds.

In [ ]:
fig = viz.plot_text_overview(corpus)
plt.show()

In [ ]:
# Peek at a couple of raw unstructured documents
for d in corpus:
    if d.source_type == "sop":
        print("SOP DOC:\n", d.text[:400], "...\n")
        break
for d in corpus:
    if d.source_type == "work_order":
        print("WORK ORDER DOC:\n", d.text[:400], "...")
        break

## 6. Interactive exploration with pandas

Load the structured entities into DataFrames to slice and dice. Handy for ad-hoc questions and for sanity-checking the generator.

In [ ]:
import pandas as pd

components = pd.DataFrame(dataset["components"])
work_orders = pd.DataFrame(dataset["work_orders"])
telemetry = pd.DataFrame(dataset["telemetry"])

print("Components by type and status:")
display(pd.crosstab(components["type"], components["status"]))

print("\nComponents per platform:")
display(components["asset_id"].value_counts())

In [ ]:
# Which components have the most maintenance history?
top = work_orders["comp_id"].value_counts().head(10)
comp_name = components.set_index("comp_id")["name"].to_dict()
top.index = [f"{comp_name.get(c, c)} ({c})" for c in top.index]
ax = top.sort_values().plot.barh(figsize=(9, 5), color="#ff7f0e")
ax.set_title("Top 10 components by work-order count")
ax.set_xlabel("work orders")
plt.tight_layout(); plt.show()

In [ ]:
# Walk the graph: what is downstream of Separator V-202, and what failure modes apply?
from graphrag.schema import NodeLabel
from graphrag.data_generation import TYPE_TO_FAILURES

sep = next(c for c in store.nodes_by_label(NodeLabel.COMPONENT)
           if "V-202" in c.properties.get("name", ""))
downstream = store.downstream(sep.id, max_depth=3)
print(f"Downstream of {sep.properties['name']} ({sep.id}):")
for n in downstream:
    fms = TYPE_TO_FAILURES.get(n.properties.get("type"), [])
    print(f"  {n.properties['name']:<28} type={n.properties['type']:<18} failure_modes={fms}")

## 7. Save all figures to disk

Render every figure to `data/figures/` (handy for the thesis document). Equivalent to running `python -m scripts.visualize` from the command line.

In [ ]:
paths = viz.save_all_figures(store, dataset, corpus, cfgmod.FIGURES_DIR)
print("Saved figures:")
for p in paths:
    print("  -", p)

---
**Next steps:** build the indexes (`python -m scripts.build_index`), then query the system (`python -m scripts.run_query --compare "..."`) or launch the dashboard (`streamlit run app/streamlit_app.py`). See the project `README.md` and `docs/methodology.md` for the full pipeline.